# **TETRIS Flow**
### 1. **TAC Generation** <br>
**Input**: C code 
- We start with C code file which we transform into TAC format. 
- Here TAC means RHS of every assigment contains only one operand, e.g., a = b + c is allowed but a = b+c+d is not allowed
- *convert_to_TAC* function converts the C file into TAC format.

In [ ]:
# and cloud of input file
import dse_pipeline 
import importlib
from dse_pipeline import convert_to_TAC, run_inliner, generator_and_cloud, run_maskedHLS, generator_and_xor_cloud
from PIL import Image # type: ignore
from IPython.display import display # type: ignore

# 1.
#  convert the test c file to tac
convert_to_TAC('/home/tapish08/Desktop/DSE-Framework/TestFiles/C_files/Boyer_Peraltas_sbox.c')


### 2. **AND Tree Generation**
- For and Tree Generation we first inline the TAC C format file.
- The inline C code preserves the dependency between the and operation present in the C code.
- This inlined file is then used to generate the **AND** tree. 

In [ ]:
# # 2. run inliner
run_inliner('/home/tapish08/Desktop/DSE-Framework/TestFiles/TAC_files/output.c','sbox')
# run_inliner(
#     input_file='/home/tapish08/Desktop/DSE-Framework/TestFiles/TAC_files/output.c', 
#     top_module='sbox',
#     script_path='/home/tapish08/Desktop/DSE-Framework/'  # <--- Force the root directory here!
# )

In [ ]:
# 3. get BOTH trees - AND tree and AND-XOR tree
print("[INFO] Generating AND tree (pure AND gates for MRLC DP)...")
and_tree = generator_and_cloud('/home/tapish08/Desktop/DSE-Framework/TestFiles/inlined_output/output.c')
print(f"[INFO] AND tree generated with {len(list(and_tree.graph.nodes()))} nodes")

print("\n[INFO] Generating AND-XOR tree (for pattern detection)...")
and_xor_tree = generator_and_xor_cloud('/home/tapish08/Desktop/DSE-Framework/TestFiles/inlined_output/output.c')
print(f"[INFO] AND-XOR tree generated")
print(f"[DEBUG] and_xor_tree type: {type(and_xor_tree)}")
print(f"[DEBUG] and_xor_tree attributes: {[a for a in dir(and_xor_tree) if not a.startswith('_')]}")

if hasattr(and_xor_tree, 'nodes_by_level'):
    print(f"[DEBUG] and_xor_tree.nodes_by_level exists with {len(and_xor_tree.nodes_by_level)} levels")
else:
    print("[DEBUG] and_xor_tree does NOT have nodes_by_level")

print("\n[INFO] Tree Strategy:")
print("  - and_tree: Used for MRLC DP algorithm (pure AND gates)")
print("  - and_xor_tree: Used for AND-XOR pattern detection")

# 4. get the image
image = Image.open('/home/tapish08/Desktop/DSE-Framework/ANDCloud/AND_tree/and_cloud.png')  # Change to your image path
display(image)

In [ ]:
# 4. get the image
image = Image.open('/home/tapish08/Desktop/DSE-Framework/ANDCloud/AND_tree/and_cloud.png')  # Change to your image path
display(image)


In [ ]:
import sqlite3
# create db if does not exist

d = 1
print(f"[INFO] : Connecting Database ... ")
conn = sqlite3.connect('gadget_info.db')
conn.execute("PRAGMA forgien_keys = ON")
cur = conn.cursor()

## 4. RUN **MLRSC** 

In [ ]:
from tabulate import tabulate



In [ ]:

from tabulate import tabulate # type: ignore


gadget_specs = {}
gadget_list = ["comar", "hpc1", "hpc2", "hpc3", "hpc2o", "hpc3o", "hpc2_swapped", "hpc2o_swapped"] if d == 1 else ["hpc1", "hpc2", "hpc3", "hpc2o", "hpc3o", "hpc2_swapped", "hpc2o_swapped "]
# gadget_list = ["hpc2"] if d == 1 else ["hpc2"]
for gadget in gadget_list:
    table = gadget.lower()
    # print(f"[INFO] : d value while populating the map : {d}")
    cur.execute(f"SELECT * FROM {table} WHERE d = ?", (d,))
    row = cur.fetchone()

    if row:
        gadget_specs[gadget] = {
            "latency": row[1],
            "randomness": row[2],
            "area": row[3]
        }
    else:
        print(f"[WARN] No entry for {gadget} with d = {d}")
if gadget_specs:
    table_data =[]
    for gadget, values in gadget_specs.items():
        table_data.append([gadget, values['latency'], values['randomness'], values['area']])
    print(f"\n[INFO]: Populated DSE MAP for {d} order")

    # from tabulate import tabulate
    print(tabulate(table_data, headers=["Gadget", "Latency", "randomness","area"], tablefmt="grid"))

# DSE.set_gadget_specs(gadget_specs)
# print(f"DSE.GADGET_gadget_specs :\n {DSE.GADGET_gadget_specs}")
conn.commit()


In [ ]:
import Gadgets
import Gadgets.hpc2_swapped
import Gadgets.hpc2o_swapped
importlib.reload(Gadgets.hpc2_swapped)
importlib.reload(Gadgets.hpc2o_swapped)
importlib.reload(Gadgets)

import DSE_algorithm.mlrc
from DSE_algorithm.mlrc import MLRC

# Force reload to pick up the arrival depth-based optimization
importlib.reload(DSE_algorithm.mlrc)

print("=" * 80)
print("MLRC with CORRECTED CRITICAL PATH LATENCY (accounts for parallelism)")
print("=" * 80)

mlrc_obj = MLRC(d=d, and_tree=and_tree, gadget_map=gadget_specs, and_xor_tree=and_xor_tree)
mlrc_obj.mlrc(target_randomness=48)
# stats = mlrc_obj.optimize_gadget_mapping_for_cost_asymmetric()


# from asymmetric_latency_optimizer import AsymmetricLatencyOptimizer

# optimizer = AsymmetricLatencyOptimizer(
#     gadget_definition=mlrc_obj.gadget_definition,   # will be modified in-place
#     nodes_by_level=mlrc_obj.nodes_by_level
# )

# flip_stats = optimizer.optimize(max_iterations=30)
# print("\n" + "=" * 80)
# print("STEP 1: INITIAL MLRC RESULTS")
# print("=" * 80)
# print(f"Total Randomness Used:  {mlrc_obj.total_randomness_after_mlrc}")
# print(f"Total Latency (Critical Path): {mlrc_obj.total_latency_after_mlrc}")
# print(f"Total Area:             {mlrc_obj.total_area_after_mlrc}")

# # =========================================================================
# # NEW: Arrival depth-based asymmetric gadget optimization (matches find_swaps.py)
# # =========================================================================
# print("\n" + "=" * 80)
# print("STEP 2: ARRIVAL DEPTH-BASED OPTIMIZATION")
# print("=" * 80)

# # Run post-MLRC optimization using arrival depth tracking:
# # - Compute how deep (in AND chain) each input is
# # - For HPC3 with different input depths: replace with HPC2
# # - Place early-arriving input on slow path, late-arriving on fast path
# # - Result: balanced timing without latency increase, randomness saved
# opt_stats = mlrc_obj.optimize_gadget_mapping_for_cost_asymmetric(
#     target_latency=mlrc_obj.total_latency_after_mlrc
# )

# # Update totals after optimization
# mlrc_obj.total_randomness_after_mlrc = opt_stats["final_randomness"]
# mlrc_obj.total_area_after_mlrc = opt_stats["final_area"]
# mlrc_obj.total_latency_after_mlrc = opt_stats["final_critical_path"]

# print("\n" + "=" * 80)
# print("STEP 3: FINAL OPTIMIZED RESULTS")
# print("=" * 80)
# print(f"Total Randomness Used:  {mlrc_obj.total_randomness_after_mlrc}")
# print(f"Total Latency (Critical Path): {mlrc_obj.total_latency_after_mlrc}")
# print(f"Total Area:             {mlrc_obj.total_area_after_mlrc}")
# print(f"\nOptimization Summary:")
# print(f"  - Order d = {d} (num_shares = {d+1})")
# print(f"  - Gadget replacements: {len(opt_stats['replacements'])}")
# print(f"  - Total randomness saved: {opt_stats['total_randomness_saved']}")
# print(f"  - Method: {opt_stats.get('method', 'unknown')} (arrival depth tracking)")

# print("\n" + "=" * 80)
# print("STEP 4: FINAL GADGET MAPPING (after optimization)")
# print("=" * 80)

final_gadget_map = mlrc_obj.get_gadget_definition()
gadget_count = {}
for expr, info in final_gadget_map.items():
    gname = info["gadget_name"]
    if gname not in gadget_count:
        gadget_count[gname] = 0
    gadget_count[gname] += 1

print(f"\nGadget Distribution:")
for gname in sorted(gadget_count.keys()):
    count = gadget_count[gname]
    randomness = mlrc_obj.gadget_map[gname]["randomness"]
    total_rand = count * randomness
    print(f"  {gname:6s}: {count:2d} nodes × {randomness} rand = {total_rand:3d} total rand")

print("\n" + "=" * 80)


# Splitting into shares and adding gadget definitions for MLRC.

In [ ]:
# create the ast for the inlined file 
import os
from pycparser import parse_file, c_ast, c_generator
import transform_CFile.ctransfromer
from transform_CFile.ctransfromer import ShareTransformer
importlib.reload(transform_CFile.ctransfromer)
from DSE_algorithm.mrlc import MRLC
import time

# Start overall timing
overall_start = time.time()
print(f"[TIMING] Pipeline started at: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(overall_start))}\n")

inlined_file = os.path.abspath(r"./TestFiles/inlined_output/output.c")
ast = parse_file(inlined_file)

print("\n[PIPELINE] Step 1: Get gadget definitions from MRLC")
step1_start = time.time()
gadget_map = mlrc_obj.get_gadget_definition()
step1_time = time.time() - step1_start
print(f"  ⏱ Step 1 completed in {step1_time:.4f}s\n")

print("[PIPELINE] Step 2: Get unique gadget definitions from MRLC")
step2_start = time.time()
unique_definition_map = mlrc_obj.get_unique_gadgets_definition()
step2_time = time.time() - step2_start

print(f"\n[RESULT] Unique gadget definitions:")
for gadget_name in sorted(unique_definition_map.keys()):
    func_code = unique_definition_map[gadget_name]
    lines = len(func_code.split('\n'))
    print(f"  ✓ {gadget_name:8} : {lines:3} lines")
print(f"  ⏱ Step 2 completed in {step2_time:.4f}s\n")

# Verify all required gadgets are present
required_gadgets = {'hpc2', 'hpc2o', 'hpc3', 'hpc3o'}
missing = required_gadgets - set(unique_definition_map.keys())
if missing:
    print(f"\n[ERROR] Missing: {missing}")
else:
    print(f"\n[SUCCESS] All 4 gadgets present!")

print(f"\n[PIPELINE] Step 3: Create ShareTransformer")
step3_start = time.time()
shareTransformer = ShareTransformer(d+1, gadget_map, unique_definition_map)
step3_time = time.time() - step3_start
print(f"  ⏱ Step 3 completed in {step3_time:.4f}s\n")

print("[PIPELINE] Step 4: Transform circuit (suppressing verbose output...)")
step4_start = time.time()
# Suppress the verbose debug output during transformation
import sys
from io import StringIO

# Temporarily redirect stdout to suppress prints
old_stdout = sys.stdout
sys.stdout = StringIO()

try:
    transformed_ast = shareTransformer.transform(ast)
finally:
    # Restore stdout
    sys.stdout = old_stdout

step4_time = time.time() - step4_start
print(f"  ⏱ Step 4 completed in {step4_time:.4f}s\n")

print("[PIPELINE] Step 5: Generate C code")
step5_start = time.time()
gen = c_generator.CGenerator()
step5_time = time.time() - step5_start
print(f"  ⏱ Step 5 completed in {step5_time:.4f}s\n")

output_path = os.path.abspath(r"./TestFiles/share_output/output.c")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("[PIPELINE] Step 6: Write output to file")
step6_start = time.time()
with open(output_path, 'w') as f:
    f.write("// === GADGET DEFINITIONS ===\n")
    for gadget_name, func_str in unique_definition_map.items():
        f.write(f"\n// --- {gadget_name} ---\n")
        f.write(func_str)
        f.write("\n\n")
    
    code = gen.visit(transformed_ast)
    f.write(code)

step6_time = time.time() - step6_start
print(f"  ⏱ Step 6 completed in {step6_time:.4f}s\n")

# Calculate overall timing
overall_end = time.time()
total_time = overall_end - overall_start

print(f"[SUCCESS] Output written to {output_path}")
print(f"\n{'='*80}")
print(f"[TIMING SUMMARY]")
print(f"{'='*80}")
print(f"  Step 1 (Get gadget definitions)     : {step1_time:8.4f}s")
print(f"  Step 2 (Get unique definitions)     : {step2_time:8.4f}s")
print(f"  Step 3 (Create ShareTransformer)    : {step3_time:8.4f}s")
print(f"  Step 4 (Transform circuit)          : {step4_time:8.4f}s  ⭐ (Longest)")
print(f"  Step 5 (Generate C code)            : {step5_time:8.4f}s")
print(f"  Step 6 (Write to file)              : {step6_time:8.4f}s")
print(f"  {'-'*80}")
print(f"  TOTAL PIPELINE TIME                 : {total_time:8.4f}s ({total_time*1000:.2f} ms)")
print(f"{'='*80}\n")


In [ ]:
import re

print("=" * 80)
print("GADGET FUNCTION CALL VERIFICATION")
print("=" * 80)

# Read the generated C code
with open(output_path, 'r') as f:
    code_content = f.read()

# Expected argument counts for each gadget
expected_args = {
    'hpc2': 7,      # a0, a1, b0, b1, &c0, &c1, r01
    'hpc2o': 9,     # a0, a1, b0, b1, w0, w1, &c0, &c1, r01
    'hpc3': 8,      # a0, a1, b0, b1, &c0, &c1, r01, p01
    'hpc3o': 10     # a0, a1, b0, b1, w0, w1, &c0, &c1, r01, p01
}

# Find all gadget calls
gadget_calls = re.findall(r'(HPC\d[a-z]*)\s*\((.*?)\);', code_content, re.IGNORECASE)

print(f"\n[INFO] Found {len(gadget_calls)} gadget function calls\n")

gadget_count = {}
incorrect_calls = []

for gadget_name, args_str in gadget_calls:
    gadget_lower = gadget_name.lower()
    arg_count = len([a.strip() for a in args_str.split(',') if a.strip()])
    
    if gadget_lower not in gadget_count:
        gadget_count[gadget_lower] = {'total': 0, 'correct': 0}
    
    gadget_count[gadget_lower]['total'] += 1
    
    expected = expected_args.get(gadget_lower, 0)
    is_correct = arg_count == expected
    
    if is_correct:
        gadget_count[gadget_lower]['correct'] += 1
    else:
        incorrect_calls.append({
            'gadget': gadget_name,
            'expected': expected,
            'actual': arg_count,
            'args': args_str[:80] + '...' if len(args_str) > 80 else args_str
        })

# Print summary
print("SUMMARY BY GADGET TYPE:")
print("-" * 80)
for gadget, stats in sorted(gadget_count.items()):
    status = "✓" if stats['correct'] == stats['total'] else "❌"
    print(f"{status} {gadget.upper():8s}: {stats['correct']:3d}/{stats['total']:3d} correct " +
          f"(expected {expected_args.get(gadget, 0)} args)")

if incorrect_calls:
    print("\n" + "=" * 80)
    print("INCORRECT FUNCTION CALLS FOUND:")
    print("=" * 80)
    for i, call in enumerate(incorrect_calls[:10], 1):  # Show first 10
        print(f"\n{i}. {call['gadget']}()")
        print(f"   Expected: {call['expected']} arguments")
        print(f"   Actual: {call['actual']} arguments")
        print(f"   Sample: {call['gadget']}({call['args']})")
    
    if len(incorrect_calls) > 10:
        print(f"\n... and {len(incorrect_calls) - 10} more incorrect calls")
else:
    print("\n" + "=" * 80)
    print("✅ ALL GADGET FUNCTION CALLS ARE CORRECT!")
    print("=" * 80)


## Run **MaskedHLS**

In [ ]:
top_module = 'sbox'
input_file = os.path.abspath("./TestFiles/share_output/output.c")
print(f"input file for maskedHLS : {input_file}")

latency = run_maskedHLS(top_module=top_module, input_file=input_file, rtl_file='test.v', bit_width=1)



## 4. RUN **MRLSC** 

In [ ]:
import DSE_algorithm.mrlsc 
from DSE_algorithm.mrlsc import (MRLC, run_enhanced_mrlc_with_db, 
                                get_best_secure_mapping_mrlc_with_db)
import importlib
from tabulate import tabulate

importlib.reload(DSE_algorithm.mrlsc)

# Enhanced MRLC execution with database integration
print("🚀 Starting Enhanced MRLC with Database Integration")

# Your parameters
min_d = 1  # Minimum required d
target_latency = 20 # Your target latency budget
max_d = 1  # Maximum d to try

# Run the enhanced MRLC with database integration
mapping, used_d, actual_latency, success = run_enhanced_mrlc_with_db(
    and_tree=and_tree,  # Your existing and_tree
    min_d=min_d,
    target_latency=target_latency,
    max_d=max_d,
    db_path='gadget_info.db'  # Path to your database
)

if success:
    print(f"\n🎯 SUCCESS! Found mapping with d = {used_d}")
    
    # Create a mock mrlc_obj to access the mapping for table display
    class MockMRLC:
        def __init__(self, mapping, and_tree, d):
            self.gadget_definition = mapping
            self.and_tree = and_tree
            self.d = d
    
    mrlc_obj = MockMRLC(mapping, and_tree, used_d)
    
    # Display the final mapping table
    table_data = []
    for expr, gadget_info in mrlc_obj.gadget_definition.items():
        node_no = mrlc_obj.and_tree.node_map.get(expr, None)
        table_data.append((node_no, expr, gadget_info))
    
    print(f"\n=== FINAL MAPPING (d = {used_d}) ===")
    print(tabulate(table_data, headers=["Gate No", "Expression", "Gadget Call"], tablefmt="grid"))
    
    # Show final statistics
    print(f"\n📊 Final Statistics:")
    print(f"   Used d: {used_d}")
    print(f"   Actual latency used: {actual_latency}")
    print(f"   Target latency: {target_latency}")
    print(f"   Latency efficiency: {(actual_latency/target_latency)*100:.1f}%")
    print(f"   Security level: {used_d}-order masking")
    
else:
    print("❌ No feasible mapping found!")


# Splitting into shares and adding gadget definitions for MRLSC.

In [ ]:
# Splitting into shares and adding gadget definitions for MRLC.
import os
from pycparser import parse_file, c_ast, c_generator
import transform_CFile.ctransfromer
from transform_CFile.ctransfromer import ShareTransformer
import importlib
importlib.reload(transform_CFile.ctransfromer)

# Create the AST for the inlined file 
inlined_file = os.path.abspath(r"./TestFiles/inlined_output/output.c")
ast = parse_file(inlined_file)

# FIXED: Recreate the successful MRLC object
print(f"Recreating MRLC object with successful parameters: d={used_d}")

# Get the gadget specs that were used for the successful solution
from DSE_algorithm.mrlsc import create_gadget_specs_from_db_mrlc
successful_gadget_specs = create_gadget_specs_from_db_mrlc(used_d, 'gadget_info.db')

# Create the actual MRLC object with the successful parameters
actual_mrlc_obj = MRLC(used_d, and_tree, successful_gadget_specs)

# Run MRLC to populate the gadget definitions
result_mapping, result_latency = actual_mrlc_obj.mrlc_dp(target_latency)

# Now you can access the methods you need
gadget_map = actual_mrlc_obj.get_gadget_definition()
unique_definition_map = actual_mrlc_obj.get_unique_gadgets_definition()

print(f"[INFO] Gadget map retrieved with {len(gadget_map)} entries")
print(f"[INFO] Unique definitions retrieved with {len(unique_definition_map)} entries")

# Continue with your existing code
shareTransformer = ShareTransformer(used_d + 1, gadget_map, unique_definition_map)
transformed_ast = shareTransformer.transform(ast)

gen = c_generator.CGenerator()

output_path = os.path.abspath(r"./TestFiles/share_output/output.c")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    code = gen.visit(transformed_ast)  # Fixed: use transformed_ast instead of ast
    f.write(code)

print(f"[INFO]: Shared output written to {output_path}")


## Run **MaskedHLS**

In [ ]:
# run the maskedhls to get the verilog code of the masked file
 
top_module = 'sbox'
input_file = os.path.abspath("./TestFiles/share_output/output.c")
print(f"Input file for maskedHLS : {input_file}")

latency = run_maskedHLS(top_module=top_module, input_file=input_file, rtl_file='bp.v', bit_width=8)

